In [1]:
import os, yaml, sys
import torch
from einops import reduce, rearrange
import matplotlib.pyplot as plt
import numpy as np
ENV = os.getenv("MY_ENV", "dev")
# with open("../../config.yaml", "r") as f:
#     config = yaml.safe_load(f)
# paths = config[ENV]["paths"]
sys.path.append("../src")
# sys.path.append(paths["useful_stuff_path"])
from useful_stuff.image_processing.computational_models import imgANN, load_model, get_relevant_output_layers, get_activation
from useful_stuff.general_utils.utils import get_device, get_module_by_path
device = get_device()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
from dataclasses import dataclass, field

@dataclass
class Cfg:
    model_name = "vit_l_16";
    pkg = 'timm'
    input_size = 384
    pooling = 'all'
    model_url = "facebook/dinov3-vitl16-pretrain-lvd1689m"
cfg = Cfg()

In [8]:
m = imgANN(cfg.model_name, cfg.pkg,  cfg.input_size, dtype=torch.float16, attn_implementation='sdpa', repo_url=cfg.model_url)
print(m.get_layer_output_shape(m.relevant_layers[0]))
print(m.get_pooling())
m.create_forward_hook()

11:50:14 - device being used: mps
torch.Size([577, 1024])
all


({},
 {'blocks.0.mlp.fc2': <torch.utils.hooks.RemovableHandle at 0x175fcbf50>,
  'blocks.1.mlp.fc2': <torch.utils.hooks.RemovableHandle at 0x175fcbfb0>,
  'blocks.2.mlp.fc2': <torch.utils.hooks.RemovableHandle at 0x367737470>,
  'blocks.3.mlp.fc2': <torch.utils.hooks.RemovableHandle at 0x367736ff0>,
  'blocks.4.mlp.fc2': <torch.utils.hooks.RemovableHandle at 0x367737d10>,
  'blocks.5.mlp.fc2': <torch.utils.hooks.RemovableHandle at 0x367737f50>,
  'blocks.6.mlp.fc2': <torch.utils.hooks.RemovableHandle at 0x367737c50>,
  'blocks.7.mlp.fc2': <torch.utils.hooks.RemovableHandle at 0x367737590>,
  'blocks.8.mlp.fc2': <torch.utils.hooks.RemovableHandle at 0x3677374d0>,
  'blocks.9.mlp.fc2': <torch.utils.hooks.RemovableHandle at 0x367737770>,
  'blocks.10.mlp.fc2': <torch.utils.hooks.RemovableHandle at 0x367736c90>,
  'blocks.11.mlp.fc2': <torch.utils.hooks.RemovableHandle at 0x3677362d0>,
  'blocks.12.mlp.fc2': <torch.utils.hooks.RemovableHandle at 0x367736630>,
  'blocks.13.mlp.fc2': <torch.

In [7]:
from torchvision import transforms
from PIL import Image

# Load a sample image and preprocess it for AlexNet
import urllib.request

# Download a sample image
url = "https://www.python.org/static/community_logos/python-logo.png"
urllib.request.urlretrieve(url, "sample_image.jpg")
image = Image.open("sample_image.jpg")

# Define preprocessing for AlexNet (ImageNet normalization)
preprocess = transforms.Compose([
    transforms.Resize((cfg.input_size, cfg.input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Preprocess and add batch dimension
input_tensor = preprocess(image).unsqueeze(0).to(device)

# Forward pass
with torch.no_grad():
    output = m.model(input_tensor)



# Check captured features
for layer_name, feature in m.features.items():
    print(f"{layer_name}: {feature.shape}")

layer.0.mlp.down_proj: torch.Size([1, 594944])
layer.1.mlp.down_proj: torch.Size([1, 594944])
layer.2.mlp.down_proj: torch.Size([1, 594944])
layer.3.mlp.down_proj: torch.Size([1, 594944])
layer.4.mlp.down_proj: torch.Size([1, 594944])
layer.5.mlp.down_proj: torch.Size([1, 594944])
layer.6.mlp.down_proj: torch.Size([1, 594944])
layer.7.mlp.down_proj: torch.Size([1, 594944])
layer.8.mlp.down_proj: torch.Size([1, 594944])
layer.9.mlp.down_proj: torch.Size([1, 594944])
layer.10.mlp.down_proj: torch.Size([1, 594944])
layer.11.mlp.down_proj: torch.Size([1, 594944])
layer.12.mlp.down_proj: torch.Size([1, 594944])
layer.13.mlp.down_proj: torch.Size([1, 594944])
layer.14.mlp.down_proj: torch.Size([1, 594944])
layer.15.mlp.down_proj: torch.Size([1, 594944])
layer.16.mlp.down_proj: torch.Size([1, 594944])
layer.17.mlp.down_proj: torch.Size([1, 594944])
layer.18.mlp.down_proj: torch.Size([1, 594944])
layer.19.mlp.down_proj: torch.Size([1, 594944])
layer.20.mlp.down_proj: torch.Size([1, 594944])
la

In [ ]:
m.extract_features(input_tensor, 'fx')
m.extract_features(input_tensor, 'hook')

{'blocks.0.mlp.fc2': tensor([[ 0.0366,  0.0288, -0.1139,  ...,  0.7707, -0.3836, -0.3004]],
        device='mps:0'),
 'blocks.1.mlp.fc2': tensor([[-0.0807,  0.1551, -0.0118,  ...,  0.1646, -0.0499, -0.2246]],
        device='mps:0'),
 'blocks.2.mlp.fc2': tensor([[-0.0542,  0.0410, -0.0533,  ..., -0.1117, -0.0067, -0.0886]],
        device='mps:0'),
 'blocks.3.mlp.fc2': tensor([[-0.0806,  0.0712, -0.0035,  ...,  0.4197, -0.0251, -0.2210]],
        device='mps:0'),
 'blocks.4.mlp.fc2': tensor([[-0.0670, -0.0151, -0.0111,  ...,  0.0622, -0.0952, -0.6685]],
        device='mps:0'),
 'blocks.5.mlp.fc2': tensor([[-0.0651,  0.0574,  0.0204,  ..., -0.4448,  0.2042, -0.6559]],
        device='mps:0'),
 'blocks.6.mlp.fc2': tensor([[-0.0857,  0.0258, -0.0038,  ..., -0.5730, -0.1446, -0.1079]],
        device='mps:0'),
 'blocks.7.mlp.fc2': tensor([[-0.0431,  0.0525,  0.0226,  ..., -0.4092, -0.5167,  0.2335]],
        device='mps:0'),
 'blocks.8.mlp.fc2': tensor([[-3.3052e-02,  1.6188e-03,  1.2517e